<a href="https://colab.research.google.com/github/lsmee/Dimensions-Departmental-Reporting/blob/main/Benchmarking_Report_Part_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%javascript
function KeepAlive() {
  document.querySelector("#top-toolbar > colab-connect-button")
    .shadowRoot.querySelector("#connect").click();
}
setInterval(KeepAlive, 60000);

<IPython.core.display.Javascript object>

In [ ]:
# Dimensions API Institutional Benchmarking Script

import subprocess, sys

print("Setting up environment...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "dimcli", "pandas", "openpyxl"], check=True)

import dimcli
from dimcli.utils import *
import pandas as pd
import numpy as np
from getpass import getpass
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── Login ─────────────────────────────────────────────────────────────────────
print("\nDimcli - Dimensions API Client")
api_key = getpass("Paste Dimensions API Key: ")
dimcli.login(key=api_key, endpoint="https://app.dimensions.ai")
dsl = dimcli.Dsl()

# ── Inputs ────────────────────────────────────────────────────────────────────
start_year = input("Start Year (e.g., 2020): ").strip()
end_year   = input("End Year   (e.g., 2025): ").strip()
years      = list(range(int(start_year), int(end_year) + 1))

print("\n" + "="*60)
print("INSTITUTIONAL BENCHMARKING")
print("="*60 + "\n")

grid_ids_input = input("Enter GRID IDs (comma separated): ")
grid_ids = [g.strip() for g in grid_ids_input.split(',') if g.strip()]
if not grid_ids:
    print("❌ No valid GRID IDs"); sys.exit(1)
print(f"✓ {len(grid_ids)} institutions")

print("\nℹ️  Enter 4-digit ANZSRC FoR codes (e.g., 3103, 3101)")
for_input = input("Enter FoR Codes (comma separated): ")
for_codes = [f.strip() for f in for_input.split(',') if f.strip()]
if not for_codes:
    print("❌ No valid FoR codes"); sys.exit(1)
print(f"✓ {len(for_codes)} FoR codes: {', '.join(for_codes)}")

for_conditions_name = " or ".join([f'category_for.name ~ "{c}"' for c in for_codes])

# ── Detect country from GRID IDs via Dimensions ───────────────────────────────
print("\n🌍 Detecting countries for provided GRID IDs...")
detected_countries = {}
for grid_id in grid_ids:
    try:
        res = dsl.query(f"""
            search organizations where id = "{grid_id}"
            return organizations[id+name+country_name+country_code]
            limit 1
        """)
        if res and res.data:
            df_org = res.as_dataframe()
            if df_org is not None and len(df_org) > 0:
                code_val = df_org['country_code'].iloc[0] if 'country_code' in df_org.columns else None
                name_val = df_org['country_name'].iloc[0] if 'country_name' in df_org.columns else None
                org_name = df_org['name'].iloc[0] if 'name' in df_org.columns else grid_id
                detected_countries[grid_id] = {'code': code_val, 'name': name_val, 'org': org_name}
                print(f"  {grid_id} ({org_name}): {name_val} [{code_val}]")
            else:
                print(f"  ⚠ No org data for {grid_id}")
                detected_countries[grid_id] = {'code': None, 'name': None, 'org': grid_id}
    except Exception as e:
        print(f"  ⚠ Error looking up {grid_id}: {e}")
        detected_countries[grid_id] = {'code': None, 'name': None, 'org': grid_id}

# ── Determine benchmark country ───────────────────────────────────────────────
unique_codes = set(
    v['code'] for v in detected_countries.values()
    if v['code'] is not None
)

if len(unique_codes) == 1:
    benchmark_country_code = unique_codes.pop()
    benchmark_country_name = next(
        v['name'] for v in detected_countries.values()
        if v['code'] == benchmark_country_code
    )
    print(f"\n✓ All institutions are from {benchmark_country_name} [{benchmark_country_code}]")
    print(f"  National benchmark will use: {benchmark_country_code}")
elif len(unique_codes) > 1:
    print(f"\n⚠ Institutions span multiple countries: {', '.join(sorted(unique_codes))}")
    print("  Options:")
    print("  1. Enter a specific country code to use as the national benchmark (e.g. US, GB, AU)")
    print("  2. Press Enter to skip the national benchmark (Global only)")
    country_input = input("  Choice: ").strip().upper()
    if country_input:
        benchmark_country_code = country_input
        # Try to find the full name from detected data, otherwise use the code
        benchmark_country_name = next(
            (v['name'] for v in detected_countries.values() if v['code'] == country_input),
            country_input
        )
        print(f"  ✓ Using {benchmark_country_name} [{benchmark_country_code}] as national benchmark")
    else:
        benchmark_country_code = None
        benchmark_country_name = None
        print("  ✓ Skipping national benchmark — Global only")
else:
    print("\n⚠ Could not detect any country codes. Skipping national benchmark.")
    benchmark_country_code = None
    benchmark_country_name = None

# Label used throughout the report (e.g. "ALL AUSTRALIA", "ALL UNITED STATES")
national_label = f"ALL {benchmark_country_name.upper()}" if benchmark_country_name else None

# ── Helper: get total count ───────────────────────────────────────────────────
def get_total_count(res):
    try:
        val = res.count_total
        if val is not None: return int(val)
    except: pass
    try:
        val = res.json.get('_stats', {}).get('total_count')
        if val is not None: return int(val)
    except: pass
    return None

# ── Helper: geo mean FCR ─────────────────────────────────────────────────────
def geo_mean_fcr(series):
    vals = pd.to_numeric(series, errors='coerce').dropna().values
    vals = vals[vals >= 0]
    if len(vals) == 0: return 0.0
    return float(np.exp(np.mean(np.log(vals + 1))) - 1)

# ── Helper: paginated institution query ───────────────────────────────────────
def get_all_publications(grid_id, for_conditions, start_year, end_year):
    all_data, skip, batch = [], 0, 1000
    while True:
        q = f"""
        search publications
            where research_orgs.id = "{grid_id}"
            and ({for_conditions})
            and year in [{start_year}:{end_year}]
        return publications[id+title+year+times_cited+field_citation_ratio+category_for+research_orgs]
        limit {batch} skip {skip}
        """
        try:
            res = dsl.query(q)
            if res and hasattr(res, 'data') and res.data:
                df_b = res.as_dataframe()
                if df_b is not None and len(df_b) > 0:
                    all_data.append(df_b)
                    if len(df_b) < batch: break
                    skip += batch
                    print(f"     {skip} retrieved...")
                else: break
            else: break
        except Exception as e:
            print(f"     Error at skip {skip}: {e}"); break
    return pd.concat(all_data, ignore_index=True) if all_data else None

# ── Helper: paginated national publications ───────────────────────────────────
def get_national_publications(for_conditions, start_year, end_year, country_code):
    all_data, skip, batch = [], 0, 1000
    while True:
        q = f"""
        search publications
            where ({for_conditions})
            and year in [{start_year}:{end_year}]
            and research_org_countries = "{country_code}"
        return publications[id+year+times_cited+field_citation_ratio+category_for]
        limit {batch} skip {skip}
        """
        try:
            res = dsl.query(q)
            if res and hasattr(res, 'data') and res.data:
                df_b = res.as_dataframe()
                if df_b is not None and len(df_b) > 0:
                    all_data.append(df_b)
                    skip += batch
                    print(f"     {skip} {country_code} pubs retrieved...")
                    if len(df_b) < batch:
                        print("     ✓ Complete!")
                        break
                else: break
            else: break
        except Exception as e:
            print(f"     Error at skip {skip}: {e}"); break
    return pd.concat(all_data, ignore_index=True) if all_data else None

# ── Helper: citation threshold + total count ─────────────────────────────────
def get_threshold_with_count(year_or_range, percentile_pct, for_conditions, country_filter=None):
    country_clause = f'and research_org_countries = "{country_filter}"' if country_filter else ''
    yr_clause = (f'year in [{year_or_range[0]}:{year_or_range[1]}]'
                 if isinstance(year_or_range, tuple) else f'year = {year_or_range}')
    try:
        res   = dsl.query(f"""
            search publications
                where {yr_clause} and ({for_conditions}) {country_clause}
            return publications[id] limit 1
        """)
        total = get_total_count(res)
        if not total: return None, None
    except: return None, None

    D      = total * (percentile_pct / 100.0)
    skip_n = int(max(round(D / 1000.0 - 1, 0), 0)) * 1000
    try:
        res2 = dsl.query(f"""
            search publications
                where {yr_clause} and ({for_conditions}) {country_clause}
            return publications[times_cited]
            sort by times_cited desc
            limit 1000 skip {skip_n}
        """)
        if res2 and res2.data:
            df2 = res2.as_dataframe()
            if df2 is not None and len(df2) > 0:
                idx = max(0, min(int(round(D)) - skip_n - 1, len(df2) - 1))
                return int(df2['times_cited'].iloc[idx]), total
    except: pass
    return None, total

# ── Helper: count global pubs above a citation threshold ─────────────────────
def count_global_above_threshold(year, threshold, for_conditions):
    if threshold is None: return None
    try:
        res = dsl.query(f"""
            search publications
                where year = {year}
                and ({for_conditions})
                and times_cited > {threshold}
            return publications[id] limit 1
        """)
        return get_total_count(res)
    except: return None

# ── Helper: calculate national + global thresholds ───────────────────────────
def get_all_thresholds_both_scopes(years, for_conditions, label="", national_country=None):
    tiers    = {'Top 1%': 1, 'Top 5%': 5, 'Top 10%': 10}
    national = {tier: {} for tier in tiers}
    global_  = {tier: {} for tier in tiers}
    global_counts = {}
    global_pcts   = {year: {} for year in years}

    scope_label = national_country if national_country else "N/A"
    print(f"     Calculating per-year thresholds [{scope_label} + Global] for {label}...")
    for year in years:
        first_tier   = True
        total_global = None
        for tier, pct in tiers.items():
            if national_country:
                national[tier][year], _ = get_threshold_with_count(
                    year, pct, for_conditions, country_filter=national_country)
            else:
                national[tier][year] = None
            threshold, total = get_threshold_with_count(
                year, pct, for_conditions, country_filter=None)
            global_[tier][year] = threshold
            if first_tier and total is not None:
                global_counts[year] = total
                total_global = total
                first_tier = False
            elif total_global is None and total is not None:
                total_global = total

        for tier in tiers:
            threshold    = global_[tier][year]
            total_global = global_counts.get(year)
            if threshold is not None and total_global:
                above = count_global_above_threshold(year, threshold, for_conditions)
                if above is not None:
                    global_pcts[year][tier] = f"{above / total_global * 100:.1f}%"
                else:
                    global_pcts[year][tier] = "N/A"
            else:
                global_pcts[year][tier] = "N/A"

    return national, global_, global_counts, global_pcts

# ── Helper: filter dataframe by FoR ID set ───────────────────────────────────
def filter_by_for_ids(df, for_ids_set):
    if not for_ids_set or df is None or len(df) == 0: return pd.DataFrame()
    mask = df['category_for'].apply(
        lambda x: any(item.get('id', '') in for_ids_set
                      for item in x if isinstance(item, dict))
        if isinstance(x, list) else False
    )
    return df[mask].copy()

# ── Helper: compute summary metrics ──────────────────────────────────────────
def compute_metrics(df, name):
    n   = len(df)
    cit = int(df['times_cited'].sum())
    pubs_with_fcr = int(df['field_citation_ratio'].notna().sum()) if 'field_citation_ratio' in df.columns else 0
    return {
        'Institution':             name,
        'Publications':            n,
        'Citations':               cit,
        'Citations per paper':     round(cit / n, 2) if n else 0,
        'Publications with FCR':   pubs_with_fcr,
        'FCR (geo mean)':          round(geo_mean_fcr(df['field_citation_ratio']), 2) if 'field_citation_ratio' in df.columns else 0.0,
        '% Publications with FCR': f"{(pubs_with_fcr / n * 100):.1f}%" if n else "0.0%",
    }

# ── Helper: write formatted Summary sheet ────────────────────────────────────
def write_summary_sheet(writer, sheet_name, summary_data, for_code, start_year, end_year):
    ws = writer.book.create_sheet(title=sheet_name)

    hdr_fill  = PatternFill("solid", fgColor="2E4057")
    hdr_font  = Font(name='Arial', bold=True, color="FFFFFF", size=10)
    aus_fill  = PatternFill("solid", fgColor="D9E1F2")
    aus_font  = Font(name='Arial', bold=True, size=10)
    alt_fill  = PatternFill("solid", fgColor="F5F7FA")
    norm_font = Font(name='Arial', size=10)
    thin      = Side(style='thin', color='CCCCCC')
    border    = Border(left=thin, right=thin, top=thin, bottom=thin)
    center    = Alignment(horizontal='center', vertical='center')
    left      = Alignment(horizontal='left',   vertical='center')

    cols       = ['Institution', 'Publications', 'Citations', 'Citations per paper',
                  'Publications with FCR', 'FCR (geo mean)', '% Publications with FCR']
    col_widths = [35, 15, 15, 22, 24, 16, 26]

    r = 1
    title = ws.cell(r, 1, f"Summary — {for_code}  |  {start_year}–{end_year}")
    title.font = Font(name='Arial', bold=True, size=13)
    ws.merge_cells(start_row=r, start_column=1, end_row=r, end_column=len(cols))
    ws.row_dimensions[r].height = 28
    r += 2

    for col_i, (h, w) in enumerate(zip(cols, col_widths), 1):
        c = ws.cell(r, col_i, h)
        c.font = hdr_font; c.fill = hdr_fill
        c.alignment = left if col_i == 1 else center
        c.border = border
        ws.column_dimensions[get_column_letter(col_i)].width = w
    ws.row_dimensions[r].height = 20
    r += 1

    for i, row_data in enumerate(summary_data):
        is_national = national_label and row_data.get('Institution') == national_label
        row_fill    = aus_fill if is_national else (alt_fill if i % 2 == 0 else PatternFill())
        row_font    = aus_font if is_national else norm_font
        row_border  = Border(left=thin, right=thin, bottom=thin,
                             top=Side(style='medium', color='2E4057')) if is_national else border
        for col_i, key in enumerate(cols, 1):
            c = ws.cell(r, col_i, row_data.get(key, ''))
            c.font = row_font; c.fill = row_fill
            c.alignment = left if col_i == 1 else center
            c.border = row_border
        ws.row_dimensions[r].height = 18
        r += 1

# ── Helper: write Top Cited sheet ────────────────────────────────────────────
def write_top_cited_sheet(writer, sheet_name, inst_rows, national_thresholds, global_thresholds,
                          global_counts, global_pcts, df_national_subset, years, start_year, end_year,
                          benchmark_country_code, national_label):
    ws = writer.book.create_sheet(title=sheet_name)

    hdr_fill  = PatternFill("solid", fgColor="2E4057")
    nat_hdr   = PatternFill("solid", fgColor="1F4E79")
    gl_hdr    = PatternFill("solid", fgColor="375623")
    yr_fill   = PatternFill("solid", fgColor="D6DCE4")
    nat_fill  = PatternFill("solid", fgColor="D9E1F2")
    gl_fill   = PatternFill("solid", fgColor="E2EFDA")
    alt_fill  = PatternFill("solid", fgColor="F5F7FA")

    title_font = Font(name='Arial', bold=True, size=13)
    yr_font    = Font(name='Arial', bold=True, size=12)
    hdr_font   = Font(name='Arial', bold=True, color="FFFFFF", size=10)
    nat_font   = Font(name='Arial', bold=True, size=10)
    gl_font    = Font(name='Arial', bold=True, size=10)
    norm_font  = Font(name='Arial', size=10)

    thin   = Side(style='thin',   color='CCCCCC')
    medium = Side(style='medium', color='2E4057')
    gl_med = Side(style='medium', color='375623')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    center = Alignment(horizontal='center', vertical='center')
    left   = Alignment(horizontal='left',   vertical='center')

    # Build column headers dynamically based on whether national benchmark exists
    fixed_cols  = ['Institution', 'Total Pubs (year)']
    nat_label_short = benchmark_country_code if benchmark_country_code else ''
    if benchmark_country_code:
        nat_cols    = [f'{nat_label_short} Top 1%', f'{nat_label_short} Top 5%', f'{nat_label_short} Top 10%']
        nat_header  = f'{nat_label_short} Benchmark'
    else:
        nat_cols    = []
        nat_header  = None
    global_cols = ['Global Top 1%', 'Global Top 5%', 'Global Top 10%']
    all_cols    = fixed_cols + nat_cols + global_cols

    base_widths = [35, 18]
    nat_widths  = [13, 13, 13] if benchmark_country_code else []
    gl_widths   = [15, 15, 15]
    col_widths  = base_widths + nat_widths + gl_widths

    for col_i, w in enumerate(col_widths, 1):
        ws.column_dimensions[get_column_letter(col_i)].width = w

    for_code = sheet_name.split('(')[-1].replace(')', '').strip()
    r = 1
    t = ws.cell(r, 1, f"Top Cited — {for_code}  |  {start_year}–{end_year}")
    t.font = title_font
    ws.merge_cells(start_row=r, start_column=1, end_row=r, end_column=len(all_cols))
    ws.row_dimensions[r].height = 28
    r += 2

    def pct(df, threshold, year):
        if threshold is None: return "N/A"
        yr_pubs = df[df['year'].astype('Int64') == year]
        if len(yr_pubs) == 0: return "0.0%"
        return f"{(yr_pubs['times_cited'] > threshold).sum() / len(yr_pubs) * 100:.1f}%"

    nat_start = len(fixed_cols) + 1
    gl_start  = nat_start + len(nat_cols)

    for year in years:
        # Year banner
        c = ws.cell(r, 1, str(year))
        c.font = yr_font; c.fill = yr_fill
        c.alignment = center; c.border = border
        ws.merge_cells(start_row=r, start_column=1, end_row=r, end_column=len(all_cols))
        ws.row_dimensions[r].height = 24
        r += 1

        # Group header row
        for col_i in range(1, len(fixed_cols) + 1):
            c = ws.cell(r, col_i, '')
            c.fill = hdr_fill; c.border = border

        if benchmark_country_code:
            c = ws.cell(r, nat_start, nat_header)
            c.font = hdr_font; c.fill = nat_hdr; c.alignment = center; c.border = border
            ws.merge_cells(start_row=r, start_column=nat_start,
                           end_row=r, end_column=nat_start + len(nat_cols) - 1)
            for col_i in range(nat_start + 1, nat_start + len(nat_cols)):
                ws.cell(r, col_i).fill = nat_hdr; ws.cell(r, col_i).border = border

        c = ws.cell(r, gl_start, 'Global Benchmark')
        c.font = hdr_font; c.fill = gl_hdr; c.alignment = center; c.border = border
        ws.merge_cells(start_row=r, start_column=gl_start,
                       end_row=r, end_column=gl_start + len(global_cols) - 1)
        for col_i in range(gl_start + 1, gl_start + len(global_cols)):
            ws.cell(r, col_i).fill = gl_hdr; ws.cell(r, col_i).border = border
        ws.row_dimensions[r].height = 18
        r += 1

        # Column label row
        for col_i, h in enumerate(all_cols, 1):
            fill = (nat_hdr if benchmark_country_code and nat_start <= col_i < gl_start
                    else gl_hdr if col_i >= gl_start
                    else hdr_fill)
            c = ws.cell(r, col_i, h)
            c.font = hdr_font; c.fill = fill
            c.alignment = left if col_i == 1 else center
            c.border = border
        ws.row_dimensions[r].height = 18
        r += 1

        # Thresholds for this year
        nat_t1  = national_thresholds['Top 1%'].get(year)  if benchmark_country_code else None
        nat_t5  = national_thresholds['Top 5%'].get(year)  if benchmark_country_code else None
        nat_t10 = national_thresholds['Top 10%'].get(year) if benchmark_country_code else None
        gl_t1   = global_thresholds['Top 1%'].get(year)
        gl_t5   = global_thresholds['Top 5%'].get(year)
        gl_t10  = global_thresholds['Top 10%'].get(year)

        # Institution rows
        for i, (_, df_inst, name) in enumerate(inst_rows):
            yr_pubs  = df_inst[df_inst['year'].astype('Int64') == year]
            row_fill = alt_fill if i % 2 == 0 else PatternFill()
            vals = [name, len(yr_pubs)]
            if benchmark_country_code:
                vals += [pct(df_inst, nat_t1, year), pct(df_inst, nat_t5, year), pct(df_inst, nat_t10, year)]
            vals += [pct(df_inst, gl_t1, year), pct(df_inst, gl_t5, year), pct(df_inst, gl_t10, year)]
            for col_i, val in enumerate(vals, 1):
                c = ws.cell(r, col_i, val)
                c.font = norm_font; c.fill = row_fill
                c.alignment = left if col_i == 1 else center
                c.border = border
            ws.row_dimensions[r].height = 18
            r += 1

        # ALL [COUNTRY] row
        if benchmark_country_code and df_national_subset is not None and len(df_national_subset) > 0:
            yr_nat     = df_national_subset[df_national_subset['year'].astype('Int64') == year]
            nat_border = Border(left=thin, right=thin, bottom=thin, top=medium)
            nat_vals   = [national_label, len(yr_nat)]
            nat_vals  += [pct(df_national_subset, nat_t1,  year),
                          pct(df_national_subset, nat_t5,  year),
                          pct(df_national_subset, nat_t10, year)]
            nat_vals  += ['—', '—', '—']
            for col_i, val in enumerate(nat_vals, 1):
                c = ws.cell(r, col_i, val)
                c.font = nat_font; c.fill = nat_fill
                c.alignment = left if col_i == 1 else center
                c.border = nat_border
            ws.row_dimensions[r].height = 18
            r += 1

        # ALL GLOBAL row
        yr_pcts   = global_pcts.get(year, {})
        gl_count  = global_counts.get(year)
        gl_border = Border(left=thin, right=thin, bottom=thin, top=gl_med)
        gl_vals   = ['ALL GLOBAL', gl_count if gl_count else '—']
        if benchmark_country_code:
            gl_vals += ['—', '—', '—']
        gl_vals += [yr_pcts.get('Top 1%', 'N/A'),
                    yr_pcts.get('Top 5%', 'N/A'),
                    yr_pcts.get('Top 10%', 'N/A')]
        for col_i, val in enumerate(gl_vals, 1):
            c = ws.cell(r, col_i, val)
            c.font = gl_font; c.fill = gl_fill
            c.alignment = left if col_i == 1 else center
            c.border = gl_border
        ws.row_dimensions[r].height = 18
        r += 1

        r += 2  # gap before next year

# ════════════════════════════════════════════════════════════
# DATA FETCHING
# ════════════════════════════════════════════════════════════
print("\n--- QUERYING INSTITUTIONAL PUBLICATIONS ---")
institution_data, all_publications = {}, []

for grid_id in grid_ids:
    print(f"\nQuerying {grid_id}...")
    df_inst = get_all_publications(grid_id, for_conditions_name, start_year, end_year)
    if df_inst is not None and len(df_inst) > 0:
        print(f"  ✓ {len(df_inst)} publications")
        df_inst['year'] = pd.to_numeric(df_inst['year'], errors='coerce').astype('Int64')
        inst_name = grid_id
        for orgs in df_inst['research_orgs']:
            if isinstance(orgs, list):
                for org in orgs:
                    if org.get('id') == grid_id:
                        inst_name = org.get('name', grid_id); break
                if inst_name != grid_id: break
        # Use detected org name from earlier lookup if available
        if grid_id in detected_countries and detected_countries[grid_id]['org'] != grid_id:
            inst_name = detected_countries[grid_id]['org']
        df_inst['primary_institution'] = inst_name
        df_inst['primary_grid_id']     = grid_id
        institution_data[grid_id] = {'name': inst_name, 'dataframe': df_inst}
        all_publications.append(df_inst)
    else:
        print(f"  ⚠ No publications found")

if not all_publications:
    print("\n❌ No publications found"); sys.exit(0)

df_all = pd.concat(all_publications, ignore_index=True)

# ── Resolve 5-digit FoR IDs ───────────────────────────────────────────────────
print("\n✓ Resolving FoR IDs...")
for_id_map     = {code: set() for code in for_codes}
rows_with_for  = df_all[df_all['category_for'].apply(lambda x: isinstance(x, list))]
exploded       = rows_with_for['category_for'].explode()
exploded_dicts = exploded[exploded.apply(lambda x: isinstance(x, dict))]
for_items      = pd.DataFrame(exploded_dicts.tolist())

if 'id' in for_items.columns and 'name' in for_items.columns:
    for_items = for_items[for_items['id'].notna() & for_items['name'].notna()]
    for code in for_codes:
        matched = for_items[for_items['name'].str.startswith(code, na=False)]['id'].unique()
        for_id_map[code] = set(matched)

for_id_map_list = {code: list(ids) for code, ids in for_id_map.items()}
print("  FoR ID mapping:")
for code, ids in for_id_map_list.items():
    print(f"    {code}: {ids}")

for code in for_codes:
    if not for_id_map_list[code]:
        print(f"  ⚠ No IDs for {code} — querying Dimensions...")
        try:
            res = dsl.query(f"""
                search publications
                    where category_for.name ~ "{code}"
                    and year in [{start_year}:{end_year}]
                return publications[category_for] limit 100
            """)
            if res and res.data:
                df_tmp       = res.as_dataframe()
                tmp_exploded = df_tmp['category_for'].explode()
                tmp_dicts    = tmp_exploded[tmp_exploded.apply(lambda x: isinstance(x, dict))]
                tmp_items    = pd.DataFrame(tmp_dicts.tolist())
                if 'id' in tmp_items.columns and 'name' in tmp_items.columns:
                    matched = tmp_items[tmp_items['name'].str.startswith(code, na=False)]['id'].dropna().unique()
                    for_id_map_list[code] = list(set(matched))
                    print(f"    Found: {for_id_map_list[code]}")
        except Exception as e:
            print(f"    Error: {e}")

for_conditions_by_code = {
    code: (" or ".join([f'category_for.id = "{fid}"' for fid in ids])
           if ids else f'category_for.name ~ "{code}"')
    for code, ids in for_id_map_list.items()
}
all_for_ids       = set(fid for ids in for_id_map_list.values() for fid in ids)
for_conditions_id = (" or ".join([f'category_for.id = "{fid}"' for fid in all_for_ids])
                     if all_for_ids else for_conditions_name)

# ── Fetch national publications ───────────────────────────────────────────────
if benchmark_country_code:
    print(f"\n📊 Fetching {benchmark_country_name} [{benchmark_country_code}] publications...")
    df_national_combined = get_national_publications(
        for_conditions_id, start_year, end_year, benchmark_country_code)
    if df_national_combined is not None:
        df_national_combined['year'] = pd.to_numeric(
            df_national_combined['year'], errors='coerce').astype('Int64')
        df_national_combined = df_national_combined.drop_duplicates(subset=['id'])
        print(f"   ✓ {len(df_national_combined):,} {benchmark_country_code} pubs")
else:
    df_national_combined = None
    print("\nℹ️  No national benchmark country — skipping national publications fetch.")

# ── Build per-FoR subsets ─────────────────────────────────────────────────────
print("\n--- BUILDING PER-FoR SUBSETS ---")
analyses = {}
for code in for_codes:
    ids_set          = set(for_id_map_list[code])
    df_for           = filter_by_for_ids(df_all, ids_set)
    df_national_for  = filter_by_for_ids(df_national_combined, ids_set) if df_national_combined is not None else pd.DataFrame()
    analyses[code] = {
        'for_cond':   for_conditions_by_code[code],
        'df_national': df_national_for if len(df_national_for) > 0 else None,
    }
    for grid_id in grid_ids:
        if grid_id not in institution_data: continue
        institution_data[grid_id][f'df_{code}'] = filter_by_for_ids(
            institution_data[grid_id]['dataframe'], ids_set)
    print(f"   FoR {code}: {len(df_for)} institutional | "
          f"{len(df_national_for) if df_national_for is not None else 0} {benchmark_country_code or 'national'} pubs")

# ════════════════════════════════════════════════════════════
# BUILD EXCEL
# ════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("CREATING EXCEL REPORT")
print("="*60)

output_file = "institutional_benchmarking_report.xlsx"
writer      = pd.ExcelWriter(output_file, engine='openpyxl')

# ── Info tab ──────────────────────────────────────────────────────────────────
print("\n📋 Info tab...")
wb      = writer.book
ws_info = wb.create_sheet(title='Info', index=0)

body_font = Font(name='Arial', size=10)
hdr_fill  = PatternFill("solid", fgColor="2E4057")
hdr_font2 = Font(name='Arial', bold=True, color="FFFFFF", size=10)
sub_fill  = PatternFill("solid", fgColor="D9E1F2")
thin2     = Side(style='thin', color='AAAAAA')
border2   = Border(left=thin2, right=thin2, top=thin2, bottom=thin2)
wrap      = Alignment(wrap_text=True, vertical='top')
center2   = Alignment(horizontal='center', vertical='center')

def info_header(ws, row, text):
    c = ws.cell(row, 1, text)
    c.font = hdr_font2; c.fill = hdr_fill
    c.alignment = center2; c.border = border2
    ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=2)

def info_row(ws, row, label, value):
    c1 = ws.cell(row, 1, label)
    c1.font = Font(name='Arial', bold=True, size=10); c1.fill = sub_fill
    c1.border = border2; c1.alignment = Alignment(vertical='top')
    c2 = ws.cell(row, 2, value)
    c2.font = body_font; c2.border = border2; c2.alignment = wrap

ws_info.column_dimensions['A'].width = 30
ws_info.column_dimensions['B'].width = 90

r = 1
ws_info.cell(r, 1, "Institutional Benchmarking Report — Methodology Notes").font = Font(name='Arial', bold=True, size=14)
ws_info.merge_cells(start_row=r, start_column=1, end_row=r, end_column=2)
r += 2

info_header(ws_info, r, "Report Parameters"); r += 1
info_row(ws_info, r, "Year Range",   f"{start_year} – {end_year}");  ws_info.row_dimensions[r].height = 18; r += 1
info_row(ws_info, r, "FoR Codes",    ', '.join(for_codes));           ws_info.row_dimensions[r].height = 18; r += 1
info_row(ws_info, r, "Institutions", '\n'.join([institution_data[g]['name'] for g in grid_ids if g in institution_data]))
ws_info.row_dimensions[r].height = max(18, 15 * len(grid_ids)); r += 1
national_benchmark_label = (f"{benchmark_country_name} [{benchmark_country_code}]"
                             if benchmark_country_code else "None (Global only)")
info_row(ws_info, r, "National Benchmark", national_benchmark_label); ws_info.row_dimensions[r].height = 18; r += 1
info_row(ws_info, r, "Data Source",  "Dimensions API (app.dimensions.ai)"); ws_info.row_dimensions[r].height = 18; r += 2

info_header(ws_info, r, "Tab Descriptions"); r += 1
info_row(ws_info, r, "All Publications",
    "Full list of all publications retrieved for all institutions across all FoR codes entered. "
    "Includes publication ID, title, year, citations, FCR, and flags indicating which FoR code(s) each paper belongs to.")
ws_info.row_dimensions[r].height = 45; r += 1
info_row(ws_info, r, "Summary (FoR code)",
    f"One row per institution plus a {national_label or 'benchmark'} row showing aggregate metrics: "
    "total publications, total citations, citations per paper, publications with FCR, FCR geo mean, and % publications with FCR.")
ws_info.row_dimensions[r].height = 45; r += 1
info_row(ws_info, r, "Top Cited (FoR code)",
    f"One table per year with {benchmark_country_code + ' Benchmark and ' if benchmark_country_code else ''}Global Benchmark columns. "
    f"{national_label + ' row shows national aggregate performance against the national threshold. ' if national_label else ''}"
    "ALL GLOBAL row shows total global publication count and calculated percentages of global pubs "
    "exceeding each global threshold (computed via API count queries).")
ws_info.row_dimensions[r].height = 60; r += 2

info_header(ws_info, r, "Top Cited Methodology"); r += 1
info_row(ws_info, r, "Step 1 — Citation threshold",
    "For each year and percentile tier (Top 1%, 5%, 10%), the script queries Dimensions for all publications "
    "in the specified FoR code for that year, sorted by citations descending. The citation count at the exact "
    "percentile rank is the threshold — e.g. for 10,000 publications, the Top 10% threshold is the citation "
    "count of the 1,000th most-cited paper.")
ws_info.row_dimensions[r].height = 60; r += 1
info_row(ws_info, r, "Step 2 — Institutional count",
    "For each institution and year, publications whose total lifetime citation count exceeds the threshold are counted. "
    "The percentage shown is that count divided by the institution's total publications for that specific year.")
ws_info.row_dimensions[r].height = 45; r += 1
info_row(ws_info, r, "ALL GLOBAL percentages",
    "Calculated via two API calls per tier: (1) total global pubs for that year, already retrieved during "
    "threshold calculation; (2) count of global pubs with citations above the threshold. "
    "The percentage is above-threshold count ÷ total. Due to rounding in threshold selection these will be "
    "close to but not always exactly 1%, 5%, 10%.")
ws_info.row_dimensions[r].height = 70; r += 1
info_row(ws_info, r, "Two benchmark pools",
    f"{benchmark_country_code + ' — thresholds derived from ' + benchmark_country_name + ' publications only (national landscape comparison).' + chr(10) + chr(10) if benchmark_country_code else ''}"
    "Global — thresholds derived from all publications worldwide (international comparison).")
ws_info.row_dimensions[r].height = 60; r += 1
info_row(ws_info, r, "Important caveat",
    "Citations are based on total lifetime citations at time of query, not end-of-year counts. "
    "More recent years show lower percentages because those papers have had less time to accumulate citations. "
    "Comparisons within a year are valid; cross-year comparisons should be made with caution.")
ws_info.row_dimensions[r].height = 60; r += 1
info_row(ws_info, r, "FCR (Field Citation Ratio)",
    "FCR = 1.0 is the field average for that year; FCR > 1.0 is above average. "
    "Geo mean is used to reduce distortion from highly-cited outliers. Values sourced directly from Dimensions.")
ws_info.row_dimensions[r].height = 45
print("   ✓ Done")

# ── All Publications tab ──────────────────────────────────────────────────────
print("\n📄 All Publications tab...")
df_export = df_all.copy()
for code in for_codes:
    ids_set = set(for_id_map_list[code])
    df_export[f'Has_{code}'] = df_export['category_for'].apply(
        lambda x, ids=ids_set: 'Yes' if (isinstance(x, list) and
            any(item.get('id','') in ids for item in x if isinstance(item, dict))) else 'No'
    )
df_export['for_codes'] = df_export['category_for'].apply(
    lambda x: ', '.join([f.get('name','') for f in x if isinstance(f, dict)]) if isinstance(x, list) else ''
)
export_cols  = ['primary_institution','primary_grid_id','id','title','year','times_cited']
export_names = ['Institution','GRID ID','Publication ID','Title','Year','Citations']
if 'field_citation_ratio' in df_export.columns:
    export_cols.append('field_citation_ratio'); export_names.append('FCR')
for code in for_codes:
    export_cols.append(f'Has_{code}'); export_names.append(f'Has {code}')
export_cols.append('for_codes'); export_names.append('All FoR Codes')
df_export[export_cols].rename(columns=dict(zip(export_cols, export_names))).to_excel(
    writer, sheet_name='All Publications', index=False)
print("   ✓ Done")

# ── Per-FoR tabs ──────────────────────────────────────────────────────────────
for code, analysis_data in analyses.items():
    df_national_subset = analysis_data['df_national']
    for_cond           = analysis_data['for_cond']

    print(f"\n{'='*60}\nCreating tabs for FoR: {code}\n{'='*60}")

    national_thresholds, global_thresholds, global_counts, global_pcts = \
        get_all_thresholds_both_scopes(
            years, for_cond, code,
            national_country=benchmark_country_code
        )

    # Summary
    print(f"📊 Summary ({code})...")
    summary_data = [
        compute_metrics(institution_data[g][f'df_{code}'], institution_data[g]['name'])
        for g in grid_ids
        if g in institution_data and len(institution_data[g][f'df_{code}']) > 0
    ]
    if df_national_subset is not None and len(df_national_subset) > 0:
        summary_data.append(compute_metrics(df_national_subset, national_label))
    write_summary_sheet(writer, f'Summary ({code})', summary_data, code, start_year, end_year)
    print("   ✓ Done")

    # Top Cited
    print(f"📈 Top Cited ({code})...")
    inst_rows = [
        (g, institution_data[g][f'df_{code}'], institution_data[g]['name'])
        for g in grid_ids
        if g in institution_data and len(institution_data[g][f'df_{code}']) > 0
    ]
    write_top_cited_sheet(
        writer,
        sheet_name            = f'Top Cited ({code})',
        inst_rows             = inst_rows,
        national_thresholds   = national_thresholds,
        global_thresholds     = global_thresholds,
        global_counts         = global_counts,
        global_pcts           = global_pcts,
        df_national_subset    = df_national_subset,
        years                 = years,
        start_year            = start_year,
        end_year              = end_year,
        benchmark_country_code = benchmark_country_code,
        national_label        = national_label,
    )
    print("   ✓ Done")

# ── Save ──────────────────────────────────────────────────────────────────────
writer.close()
print(f"\n✅ Report saved: {output_file}")
print(f"\nTabs:")
print(f"  • Info\n  • All Publications")
for code in for_codes:
    print(f"  • Summary ({code})   — institutions + {national_label or 'Global only'}")
    print(f"  • Top Cited ({code}) — single table per year, {benchmark_country_code + ' + ' if benchmark_country_code else ''}Global, calculated ALL GLOBAL row")

try:
    from google.colab import files
    files.download(output_file)
    print("✓ Downloaded!")
except ImportError:
    import os
    print(f"\n💡 File at: {os.path.abspath(output_file)}")

print("\n" + "="*60 + "\nBenchmarking complete!\n" + "="*60)

Setting up environment...

Dimcli - Dimensions API Client
Paste Dimensions API Key: ··········
Dimcli - Dimensions API Client (v1.6)
Connected to: <https://app.dimensions.ai/api/dsl> - DSL v2.14
Method: manual login
Start Year (e.g., 2020): 2020
End Year   (e.g., 2025): 2025

INSTITUTIONAL BENCHMARKING

Enter GRID IDs (comma separated): grid.1143130.35, grid.1001.0, grid.1002.3, grid.1013.3, grid.1005.4, grid.1008.9, grid.1003.2, grid.1012.2
✓ 8 institutions

ℹ️  Enter 4-digit ANZSRC FoR codes (e.g., 3103, 3101)
Enter FoR Codes (comma separated): 3211, 3202
✓ 2 FoR codes: 3211, 3202

🌍 Detecting countries for provided GRID IDs...
Returned Organizations: 1 (total = 1)
Time: 0.45s
  grid.1143130.35 (Adelaide University): Australia [AU]
Returned Organizations: 1 (total = 1)
Time: 0.73s
  grid.1001.0 (Australian National University): Australia [AU]
Returned Organizations: 1 (total = 1)
Time: 5.25s
  grid.1002.3 (Monash University): Australia [AU]
Returned Organizations: 1 (total = 1)
Time:

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Downloaded!

Benchmarking complete!
